In [18]:
import csv
import os
import re
from datetime import datetime

In [19]:
def recordObservation():
    file_exists = os.path.isfile("weather_data.csv")
    file_empty = os.path.getsize("weather_data.csv") == 0 if file_exists else True

    #read existing dates
    existing_dates = set()
    if file_exists and not file_empty:
        with open("weather_data.csv", "r") as file:
            reader = csv.reader(file)
            next(reader)
            for row in reader:
                existing_dates.add(row[0])

    # date
    while True:
        date = input("Enter date (MM-DD-YYYY): ")
        
        try:
            date_obj = datetime.strptime(date, "%m-%d-%Y")
            min_date = datetime(2025, 1, 1)
            today = datetime.today()
            
            if date_obj < min_date:
                print("**Date must be from 2025 onwards**")
            elif date_obj > today:
                print("**Date cannot be in the future**")
            elif date in existing_dates:
                print("**This date already exists**")
            else:
                break
                
        except:
            print("**Invalid format.. use MM-DD-YYYY**")

    # temp
    while True:
        temp = input("Enter temperature (C): ")
        try:
            temp_value = float(temp)
            if temp_value < -50 or temp_value > 60:
                print("**temperature must not exceed two digits**")
                continue
            break
        except:
            print("**Temperature must be a number**")

    # condition
    while True:
        cond = input("Enter condition (Sunny, Cloudy, Rainy or Windy): ")
        
        valid_conditions = ["sunny", "cloudy", "rainy", "windy"]
        if cond.lower() in valid_conditions:
            cond = cond.capitalize()
            break
        print("**Condition must be a word (e.g., sunny, rainy)**")

    # humidity
    while True:
        humidity = input("Enter humidity (%): ")
        if humidity.isdigit():
            h_value = int(humidity)
            if 0 <= h_value <= 100:
                break
            else:
                print("**Humidity must be between 0 and 100**")
        else:
            print("**Humidity must be a whole number**")

    # wind speed
    while True:
        wind_speed = input("Enter wind speed (km/h): ")
        try:
            w_value = float(wind_speed)
            if w_value < 0:
                print("**Wind speed cannot be negative**")
            elif w_value > 200:
                print("**Wind speed is unrealistically high**")
            else:
                break
        except:
            print("**Wind speed must be a number**")

    new_row = [date, temp_value, cond, h_value, w_value]

    with open("weather_data.csv", mode="a", newline="") as file:
        writer = csv.writer(file)

        if not file_exists or file_empty:
            writer.writerow(["Date", "Temperature", "Condition", "Humidity", "Wind Speed"])

        writer.writerow(new_row)

    print("-Your observation is recorded-")

In [20]:
recordObservation()

Enter date (MM-DD-YYYY):  14-4-2025


**Invalid format.. use MM-DD-YYYY**


Enter date (MM-DD-YYYY):  04-13-2026


**This date already exists**


Enter date (MM-DD-YYYY):  03-27-2026
Enter temperature (C):  24
Enter condition (Sunny, Cloudy, Rainy or Windy):  sunnny


**Condition must be a word (e.g., sunny, rainy)**


Enter condition (Sunny, Cloudy, Rainy or Windy):  windyy


**Condition must be a word (e.g., sunny, rainy)**


Enter condition (Sunny, Cloudy, Rainy or Windy):  windy
Enter humidity (%):  20
Enter wind speed (km/h):  60


-Your observation is recorded-


In [21]:
def search():
    search_date = input("Enter the date (MM-DD-YYYY): ")
    
    if not re.match(r"\d\d-\d\d-\d\d\d\d", search_date):
        print("**Invalid date format .. enter MM-DD-YYYY**")
        return

    if not os.path.isfile("weather_data.csv"):
        print("data file not found")
        return

    date_found = False

    with open("weather_data.csv", mode="r") as file:
        reader = csv.reader(file)
        header = next(reader)

        for row in reader:
            if row[0] == search_date:
                if not date_found:
                    print("\nResults:\n")
                    print(", ".join(header))
                print(", ".join(row))
                date_found = True

    if not date_found:
        print("**There are no observations for this date**")

In [24]:
search()

Enter the date (MM-DD-YYYY):  19-03-2026


**There are no observations for this date**


In [25]:
def displayTrends():
    if not os.path.isfile("weather_data.csv") or os.path.getsize("weather_data.csv") == 0:
        print("No data available")
        return

    observations = []
    with open("weather_data.csv", mode="r") as file:
        reader = csv.reader(file)
        try:
            header = next(reader)
        except StopIteration:
            print("File is empty.")
            return
        
        for row in reader:
            if not row or len(row) < 2:
                continue
                
            try:
                row_date = datetime.strptime(row[0], "%m-%d-%Y")
                temp_value = float(row[1])
                observations.append([row_date, row[0], temp_value])
            except ValueError:
                continue

    if not observations:
        print("No valid data to display.")
        return

    observations.sort(key=lambda x: x[0], reverse=True)
    print("\n--- Temperature Trends ---")
    print(f"{'Date':<12} | {'Trend'}")
    print("-" * 35)

    for obs in observations:
        date_str = obs[1]
        temp = obs[2]
        num_bars = max(1, int(abs(temp) // 2))
        bar = "|" * num_bars
        
        print(f"{date_str:<12} | {bar} {temp}°C")

In [26]:
displayTrends()


--- Temperature Trends ---
Date         | Trend
-----------------------------------
04-13-2026   | ||||||||||||||| 30.0°C
04-05-2026   | ||||||||||||| 27.0°C
04-04-2026   | |||||||||||| 24.0°C
03-27-2026   | |||||||||||| 24.0°C
03-09-2026   | ||||||||| 19.0°C
04-04-2025   | ||||||||||||||| 30.0°C
